# Get tcBF and acBF STEM Images with GPU

This notebook walks through a normal fast-acBF workflow:

1. Load a 4D-STEM dataset as `(scan_y, scan_x, detector_y, detector_x)`.
2. Initialize `BFSolver` with microscope metadata and an initial aberration guess.
3. Reconstruct tcBF and acBF images.
4. Optionally refine orientation, defocus, and aberrations.
5. Export images, probe, and detector-frame aberrations for downstream use.

Notes for users:

- `C10` is in Angstrom. In this code, positive `C10` corresponds to overfocus.
- Real-space vBF upscaling is supported with `upscale` and `upscale_method="bilinear"/"nearest"`; changing these options rebuilds the active ImageFFT cache while preserving aberration state.
- `pipeline="balanced"` (default) resolves the data path automatically. It caches ImageFFT on the compute device when it fits, falls back to host RAM, and streams without a persistent cache when needed. Use `pipeline="speed"` for small data where temporary raw materialization is acceptable, or `pipeline="memory"` for the lowest persistent memory footprint.
- Advanced overrides are `imagefft_storage="auto"/"device"/"host"/"none"`, `imagefft_fill="auto"/"precompute"/"lazy"/"on_the_fly"`, and `extractor_strategy="auto"/"device_mask"/"host_mask"/"disk_*"`. Impossible or wasteful combinations raise early with guidance.
- `basis_mode="on_the_fly"` (default) recomputes the aberration basis per chunk; use `basis_mode="precompute"` to pre-materialize it for a small speed gain at the cost of extra RAM.
- Use `frame="scan"` for images in scan coordinates. Use `frame="detector"` for PtyRAD-compatible aberrations, probe, and detector-oriented outputs.

Author: Dr. Chia-Hao Lee, chia-hao.lee@cornell.edu


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

# Run the notebook from the repository root by default.
# Change this if you copied the notebook elsewhere.
work_dir = Path(".").resolve()
%cd {work_dir}

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
print("Current working dir:", Path.cwd())
print("Output dir:", output_dir.resolve())


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from tifffile import imwrite

from ptyrad.io.handlers import load_array_from_file
from ptyrad.optics.constants import get_wavelength_ang
from ptyrad.runtime.diagnostics import print_system_info
from ptyrad.utils.image_proc import guess_radius_of_bright_field_disk

from fast_acbf import BFSolver

print_system_info()


## 1. Load the Dataset

Edit the variables in the next cell for your own data. The only strict requirement is that `dataset` ends up as a NumPy array with shape:

```python
(scan_y, scan_x, detector_y, detector_x)
```

For the demo dataset, download **Figure 4.zip** from the Zenodo link in the README, unzip it, and either place the `Figure 4/` folder in this repository or update `candidate_paths` below.


In [ ]:
# ---- User-editable dataset settings ---------------------------------------
# Demo: tBL-WSe2 Moire Phason dataset from Figure 4.zip.
# https://zenodo.org/records/15283332

file_path = "/home/cl2696/scratch/Figure 4/scan_x128_y128.raw"

scan_shape = (128, 128)      # (scan_y, scan_x)
detector_shape = (128, 128)  # (detector_y, detector_x)
raw_shape = (int(np.prod(scan_shape)), *detector_shape)

kv = 80                      # accelerating voltage in kV
max_alpha = 25               # convergence semi-angle in mrad
scan_step_size = 0.43        # scan step in Angstrom

# Initial aberration guess. Refinement cells below can improve this.
aberrations = {"C10": 80}

# Coordinate transform from raw detector/scan convention to the fast-acBF/PtyRAD convention.
# Available flags: flipud, fliplr, transpose, rotation_deg.
coord_transform = {
    "flipud": True,
    "fliplr": False,
    "transpose": False,
    "rotation_deg": 0.0,
}
# ---------------------------------------------------------------------------

raw = load_array_from_file(file_path, shape=raw_shape)
dataset = np.asarray(raw.reshape(*scan_shape, *detector_shape), dtype=np.float32)

mean_dp = dataset.mean(axis=(0, 1))
dataset /= mean_dp.max()  # Normalize so PACBED maximum is approximately 1.

wavelength = get_wavelength_ang(kv=kv)
bf_radius_px = guess_radius_of_bright_field_disk(mean_dp, thresh=0.5)
dk = max_alpha * 1e-3 / wavelength / bf_radius_px  # reciprocal-space pixel size, Ang^-1

print("Loaded:", file_path)
print("dataset shape:", dataset.shape)
print(f"electron wavelength: {wavelength:.5g} Ang")
print(f"estimated BF disk radius: {bf_radius_px:.2f} px")
print(f"dk: {dk:.5g} Ang^-1")


### Loading Other File Formats

For HDF5, Zarr, or NumPy-style arrays, replace the raw-loading block above with `load_array_from_file(...)`, then reshape to `(scan_y, scan_x, detector_y, detector_x)`. Keep the normalization and `dk` estimate unless you already know `dk` from calibrated metadata.


In [ ]:
# Example alternatives for your own data:
#
# dataset = load_array_from_file("/path/to/data.hdf5", key="dp")
# dataset = dataset.reshape(scan_y, scan_x, detector_y, detector_x).astype(np.float32)
#
# dataset = load_array_from_file("/path/to/cbed.zarr", selection=[-1])
# dataset = dataset.reshape(scan_y, scan_x, detector_y, detector_x).astype(np.float32)


## 2. Inspect the Dataset

The left panel is the PACBED/mean diffraction pattern. The right panel is a quick virtual bright-field image from the central detector pixels. This catches common mistakes such as wrong reshape order, wrong detector size, or a missing bright-field disk.


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

pacbed = dataset.mean(axis=(0, 1))
center_y, center_x = np.array(detector_shape) // 2
central_bf = dataset[:, :, center_y-4:center_y+4, center_x-4:center_x+4].sum(axis=(-2, -1))

im0 = axs[0].imshow(pacbed)
axs[0].set_title("PACBED")
axs[0].set_axis_off()
fig.colorbar(im0, ax=axs[0], shrink=0.75)

im1 = axs[1].imshow(central_bf, cmap="gray")
axs[1].set_title("Central virtual BF")
axs[1].set_axis_off()
fig.colorbar(im1, ax=axs[1], shrink=0.75)

plt.tight_layout()
plt.show()


## 3. Initialize `BFSolver`

`BFSolver` owns the 4D data, BF-mask extraction, pipeline policy, coordinate transform, and aberration state. Reconstruction calls are no-grad by default, so normal plotting/export does not retain PyTorch graphs.

The default `pipeline="balanced"` is usually the right starting point. It precomputes a persistent ImageFFT cache when device or host memory allows, and otherwise streams/recomputes chunks. Check `bf.pipeline`, `bf.imagefft_storage`, `bf.imagefft_fill`, and `bf.extractor_strategy` after construction to see the resolved route.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

bf = BFSolver(
    dataset=dataset,
    max_alpha=max_alpha - 0.3,  # Slightly shrink the BF mask to exclude edge pixels.
    scan_step_size=scan_step_size,
    dk=dk,
    wavelength=wavelength,
    max_order=2,
    aberrations=aberrations,
    coord_transform=coord_transform,
    device=device,
    pipeline="balanced",  # speed | balanced | memory
    # Optional low-level overrides:
    # imagefft_storage="auto",  # auto | device | host | none
    # imagefft_fill="auto",     # auto | precompute | lazy | on_the_fly
    # extractor_strategy="auto", # auto | device_mask | host_mask | disk_per_pixel | disk_slab | disk_scan_row
)

print(f"Using device: {device}")
print(
    "Resolved pipeline: "
    f"pipeline={bf.pipeline}, storage={bf.imagefft_storage}, "
    f"fill={bf.imagefft_fill}, extractor={bf.extractor_strategy}"
)
print(f"Extracted {int(bf.bf_mask.sum().item())} BF detector pixels")
print("Initial detector-frame aberrations:")
bf.print_aberrations(frame="detector")


In [ ]:
# Quick first look before refinement.
# acbf_algorithm options: "phase_only" (default) or "complex_inversion".
bf.plot_reconstruction(mode="tcBF", frame="scan")
bf.plot_reconstruction(mode="acBF", frame="scan")


## 4. Real-Space Upscale on Subsampled Data

This section deliberately makes a low-resolution scan with `dataset[::n, ::n]`, then reconstructs it with different real-space vBF upscale factors. Upscaling happens inside `BFPreparer` before padding/windowing and before the ImageFFT cache is built. Change `UPSCALE_DEMO_MODE` from `"tcBF"` to `"acBF"` if you want to inspect the same behavior with acBF images.

Important state behavior: getter call-site overrides such as `bf_lr.get_reconstructed_image(mode=UPSCALE_DEMO_MODE, upscale=2.0, upscale_method="bilinear")` are stateful. If the requested preparation is not already active, the solver rebuilds the prepared ImageFFT cache, preserves the aberration state, and updates public active properties such as `bf_lr.upscale`, `bf_lr.scan_shape`, `bf_lr.padded_scan_shape`, and `bf_lr.scan_step_size`. The original/native values remain available through `raw_` properties.


In [ ]:
# Fix subsampling rate, but increasing upscale factor

subsample_factor = 4
upscale_factors = [1, 2, 3, 4, 5, 6]
upscale_method = 'bilinear' # Available options are 'bilinear' and 'nearest'
recon_mode = 'tcBF' # 'tcbf' or 'acbf'
plot_individual = False

bf_lr = BFSolver(
    dataset=np.ascontiguousarray(dataset[::subsample_factor, ::subsample_factor]),
    max_alpha=max_alpha - 0.3,
    scan_step_size=scan_step_size * subsample_factor,
    dk=dk,
    wavelength=wavelength,
    max_order=2,
    aberrations=aberrations,
    coord_transform=coord_transform,
    device=device,
    pipeline="balanced",
)

imgs = []
for n in upscale_factors:
    img = bf_lr.get_reconstructed_image(mode=recon_mode, upscale=n, upscale_method=upscale_method)
    imgs.append(img.detach().cpu().numpy())

    if plot_individual:
        bf_lr.plot_reconstruction(title_str=f"{recon_mode} with original dataset size={bf_lr.raw_scan_shape} and upscale={n}", 
                              mode=recon_mode, upscale=n, upscale_method=upscale_method)

fig, axs = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
fig.suptitle(f'{recon_mode} with original dataset size={bf_lr.raw_scan_shape}, scan step size={bf_lr.raw_scan_step_size} Ang', fontsize=16)
for ax, img, n in zip(axs.ravel(), imgs, upscale_factors):
    ax.imshow(img)
    ax.set_title(f"upscale={n}")
plt.show()

In [ ]:
# Symmetrical downsample and upsample rate to test recoverability

upscale_factors = [1, 2, 3, 4, 5, 6]
upscale_method = 'bilinear' # Available options are 'bilinear' and 'nearest'
recon_mode = 'tcBF' # 'tcbf' or 'acbf'
plot_individual = False

imgs = []
for n in upscale_factors:
    # Building new solver as we're changing the dataset
    bf_lr = BFSolver(
        dataset=np.ascontiguousarray(dataset[::n, ::n]),
        max_alpha=max_alpha - 0.3,
        scan_step_size=scan_step_size * n,
        dk=dk,
        wavelength=wavelength,
        max_order=2,
        aberrations=aberrations,
        coord_transform=coord_transform,
        device=device,
        pipeline="balanced",
    )
    
    img = bf_lr.get_reconstructed_image(mode=recon_mode, upscale=n, upscale_method=upscale_method)
    imgs.append(img.detach().cpu().numpy())
    
    if plot_individual:
        bf_lr.plot_reconstruction(title_str=f"{recon_mode} with original dataset size={bf_lr.raw_scan_shape} and upscale={n}", 
                              mode=recon_mode, upscale=n, upscale_method=upscale_method)
        
fig, axs = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
fig.suptitle(f'{recon_mode} with upscale factors={upscale_factors}, upscale method={upscale_method}', fontsize=16, y=1.05)
for ax, img, n in zip(axs.ravel(), imgs, upscale_factors):
    ax.imshow(img)
    ax.set_title(f"Subsample dataset[::{n},::{n}], upscale={n}")
plt.show()


In [ ]:
# Refine aberrations with the upscaled image
n = 4

bf_lr = BFSolver(
    dataset=np.ascontiguousarray(dataset[::n, ::n]),
    max_alpha=max_alpha - 0.3,
    scan_step_size=scan_step_size * n,
    dk=dk,
    wavelength=wavelength,
    max_order=2,
    aberrations=aberrations,
    coord_transform=coord_transform,
    device=device,
    pipeline="balanced",
)

print(f"### Plot original")
bf_lr.plot_reconstruction()
bf_lr.refine_aberrations(upscale=n) # Refinment is more accurate when the dataset is heavily undersampled

# Plot refined result, note that the global upscale field has been updated by the prior call
print(f"### Plot refined with upscale = {n}")
bf_lr.plot_reconstruction()
bf_lr.refine_aberrations(upscale=n)


## 5. Optional Orientation Refinement

If you suspect there's relative flips between diffraction pattern and the scan pattern, use the orientation searches below.

- `refine_flips` checks all 8 combinations of `flipud`, `fliplr`, and `transpose`.
- `refine_scan_rotation` searches continuous scan rotation in degrees.
- `search_range=(a, b)` means literal bounds. `search_halfwidth=h` means current value plus/minus `h`.

For known datasets, leave this off and use the `coord_transform` from the data-loading cell.


In [ ]:
RUN_ORIENTATION_REFINEMENT = True # Change this to enable orientation (flips + scan rotation) refinement

if RUN_ORIENTATION_REFINEMENT:
    bf.refine_flips(
        metric="sobel",
        plot_search=True,
        mode="tcBF",
    )
    bf.refine_scan_rotation(
        search_range=(-45, 45),
        num_points=19,
        metric="sobel",
        plot_search=True,
        mode="tcBF",
    )
    bf.plot_reconstruction(mode="tcBF", frame="scan")
else:
    print("Skipping orientation refinement. Current coord_transform:")
    print(bf.coord_transform)


## 6. Refine Defocus and Aberrations

A practical workflow is:

1. Run a fast C10 line search with `refine_defocus`.
2. Run gradient-based aberration refinement with `refine_aberrations`.
3. Inspect detector-frame aberrations before exporting to PtyRAD.

`laplacian` and `sobel` are good general-purpose focus metrics. `tcBF` is pretty robust and is much faster for broad range aberration refinement. `normalized_std` is available, but acBF can have negative image mean and wild contrast changes, which makes it less reliable for AD-based refinement.


In [ ]:
# Fast C10 line search. This updates bf.ab_state and stores the best image in bf.reconstructed_image.
bf.refine_defocus(
    search_halfwidth=150,  # Ang around the current C10. Use search_range=(min_c10, max_c10) for absolute bounds.
    num_points=31,
    metric="sobel",
    method="max",
    plot_search=True,
    mode="tcBF",
)

bf.plot_reconstruction(mode="acBF", frame="scan")

In [ ]:
# Optional AD refinement. For large datasets or limited VRAM, set scan_roi to a crop (y0, y1, x0, x1)
# such as (32, 96, 32, 96); the final image is then reconstructed on the full field of view.
RUN_AD_REFINEMENT = True

if RUN_AD_REFINEMENT:
    bf.refine_aberrations(
        iters=120,
        lr=1,
        metric="sobel",
        mode="tcBF",
        scan_roi=None,
        plot_recon_every_n_iter=None,
        save_dir=None,
    )

print("Refined detector-frame aberrations:")
bf.print_aberrations(frame="detector")
bf.plot_reconstruction(mode="acBF", frame="detector")


### One-Call Coarse-to-Fine Refinement

The refactor also adds `refine_all_params`, which can run orientation/defocus search, coarse aberration refinement, fine rotation, and final aberration refinement as one ordered pipeline. This is useful when starting from a mostly unknown coordinate transform, but it does more work than the focused cells above.

This functions does the following:
1. Checks the combination of all flips/transpose, rotation angles, defocus = 792 configurations
2. Pick the best condition and refine 1st and 2nd order aberrations with AD
3. Refine the scan rotation angle with a finer line search
4. Refine aberrations with AD up to `max_order` (suggested value is 3)


In [ ]:
RUN_FULL_PARAMETER_REFINEMENT = True

if RUN_FULL_PARAMETER_REFINEMENT:
    bf.refine_all_params(
        targets=("orientation_defocus", "coarse_aberrations", "fine_rotation", "fine_aberrations"),
        metric="sobel",
        mode="tcBF",
        rotation_num_points=36,
        defocus_num_points=11,
        fine_rotation_halfwidth=5,
        fine_rotation_num_points=11,
        aberration_lr=0.1,
        aberration_iters=80,
        refinement_scan_roi=None,
    )
    bf.plot_reconstruction(mode="acBF", frame="detector", acbf_algorithm="complex_inversion")
else:
    print("Skipping refine_all_params.")


## 7. Export Images, Probe, and Aberrations

The current export pattern is explicit: call the image/probe/aberration getters and save what you need. A future `get_export()` helper is planned, but the manual version below mirrors the current API and keeps frame choices visible.


In [ ]:
tcbf_scan = bf.get_tcBF(frame="scan").detach().cpu().numpy().astype(np.float32)
acbf_scan = bf.get_acBF(frame="scan", acbf_algorithm="phase_only").detach().cpu().numpy().astype(np.float32)

probe_detector = bf.get_probe(frame="detector").detach().cpu().numpy()
aberrations_detector = bf.get_aberrations_dict(
    frame="detector",
    notation="krivanek",
    style="cartesian",
    layout="flat",
)

imwrite(output_dir / "tcbf_scan.tif", tcbf_scan)
imwrite(output_dir / "acbf_phase_only_scan.tif", acbf_scan)
np.savez(
    output_dir / "fast_acbf_export.npz",
    tcbf_scan=tcbf_scan,
    acbf_scan=acbf_scan,
    probe_detector_real=probe_detector.real.astype(np.float32),
    probe_detector_imag=probe_detector.imag.astype(np.float32),
)

print("Saved outputs to", output_dir.resolve())
print("Detector-frame aberrations for PtyRAD-style use:")
print(aberrations_detector)


## 8. Optional 3D Defocus Stack

`get_defocus_stack` reconstructs a stack over absolute `C10` values and records that axis in `bf.last_c10_stack_axis`. Use either:

- `n_layers` plus `slice_thickness`, centered on the current C10, or
- `z_top`, `z_bottom`, plus `slice_thickness`, for explicit C10 bounds.


In [ ]:
RUN_DEFOCUS_STACK = True

if RUN_DEFOCUS_STACK:
    volume_tcbf = bf.get_defocus_stack(
        mode="tcBF",
        frame="scan",
        n_layers=100,
        slice_thickness=10,
    ).detach().cpu().numpy().astype(np.float32)

    volume_acbf = bf.get_defocus_stack(
        mode="acBF",
        frame="scan",
        n_layers=100,
        slice_thickness=10,
    ).detach().cpu().numpy().astype(np.float32)

    c10_axis = bf.last_c10_stack_axis.detach().cpu().numpy().astype(np.float32)

    imwrite(output_dir / "volume_tcbf.tif", volume_tcbf)
    imwrite(output_dir / "volume_acbf.tif", volume_acbf)
    np.save(output_dir / "defocus_c10_axis_ang.npy", c10_axis)

    print("Saved defocus stack volumes and C10 axis.")
else:
    print("Skipping defocus-stack export.")


In [ ]:
def show_defocus_stack_slice(stack, c10_axis, title):
    """Show a y-center slice through a C10 stack and its 3D FFT magnitude."""
    stack_fft = np.log1p(np.abs(np.fft.fftshift(np.fft.fftn(stack))))
    shape = stack.shape

    vmin, vmax = np.percentile(stack, [1, 99])
    vmin_fft, vmax_fft = np.percentile(stack_fft, [1, 99])

    image_extent = (0, shape[-1], c10_axis[-1], c10_axis[0])
    d_c10 = c10_axis[1] - c10_axis[0]
    fft_freqs = np.fft.fftshift(np.fft.fftfreq(shape[0], d=abs(d_c10)))
    fft_extent = (0, shape[-1], fft_freqs[-1], fft_freqs[0])

    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    fig.suptitle(title)
    axs[0].imshow(stack[:, shape[-2] // 2], aspect="auto", extent=image_extent)
    axs[0].set_title("Raw display")
    axs[1].imshow(stack[:, shape[-2] // 2], aspect="auto", vmin=vmin, vmax=vmax, extent=image_extent)
    axs[1].set_title("1-99 percentile")
    axs[2].imshow(stack_fft[:, shape[-2] // 2], aspect="auto", extent=fft_extent, vmin=vmin_fft, vmax=vmax_fft)
    axs[2].set_title("3D FFT")

    axs[0].set_ylabel("C10 (Ang)")
    axs[1].set_ylabel("C10 (Ang)")
    axs[2].set_ylabel("Ang^-1")
    plt.tight_layout()
    plt.show()

if RUN_DEFOCUS_STACK:
    show_defocus_stack_slice(volume_tcbf, c10_axis, "tcBF defocus stack")
    show_defocus_stack_slice(volume_acbf, c10_axis, "acBF defocus stack")


## 9. Sanity Plots

These plots help check whether the fitted aberration state is physically plausible: the chi surface/probe phase should look smooth over the BF disk, and the shift quiver should not show unexpected discontinuities.


In [ ]:
bf.plot_chi_surface(plot_probe_phase=True)
bf.plot_shift_quiver(frame="detector");


## 10. Timing Snippet for Developers

This is optional and mainly useful when comparing `pipeline` presets, `imagefft_storage` / `imagefft_fill` choices, `basis_mode="on_the_fly"` vs. `basis_mode="precompute"`, chunk sizes, GPU models, or reconstruction algorithms.


In [ ]:
RUN_TIMING = True

if RUN_TIMING:
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    repetition = 100
    mode = "acBF"
    kwargs = {}

    t0 = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() else None
    t1 = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() else None

    if torch.cuda.is_available():
        t0.record()
        for _ in range(repetition):
            bf.reconstruct(mode=mode, **kwargs)
        t1.record()
        torch.cuda.synchronize()
        elapsed = t0.elapsed_time(t1) / 1000
    else:
        import time
        start = time.perf_counter()
        for _ in range(repetition):
            bf.reconstruct(mode=mode, **kwargs)
        elapsed = time.perf_counter() - start

    print(f"Average {mode} reconstruction time: {elapsed / repetition:.5f} s")
